# EmotWen — Step 1: Dataset Preparation

Loads, filters, and formats all training data for the journal companion fine-tune.

**Outputs saved to disk:**
- `data/sft_train/` — training set
- `data/sft_val/` — validation set  
- `data/eval_200/` — held-out evaluation set (never trained on)

Run on a free Colab CPU instance (no GPU needed for data prep).

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "datasets", "transformers", "nltk", "wandb", "tqdm"], check=True)
import nltk
nltk.download('punkt_tab', quiet=True)

In [ ]:
# ── Mount Google Drive and clone / pull repo ──────────────────────────────────
import os

# If running on Colab, mount Drive for persistence
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/emotwen-3.5-finetune'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/afonsomota/emotwen-3.5-finetune.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
except ImportError:
    REPO_DIR = os.path.dirname(os.getcwd())  # local dev

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Repo dir: {REPO_DIR}')

In [ ]:
# ── W&B login ─────────────────────────────────────────────────────────────────
import wandb
wandb.login()  # will prompt for API key on first run

In [ ]:
# ── Config overrides (edit this cell to customise) ────────────────────────────
CONFIG = {
    # W&B
    'wandb_project': 'emotwen-journal-chat',
    'run_name': 'data_prep_v1',

    # Dataset sizes (set to None to use all available)
    'max_empathetic': 15000,
    'max_daily_dialog': 6000,
    'max_go_emotions_synthetic': 5000,
    'max_counsel_chat': 2000,

    # RAG simulation: fraction of synthetic examples with journal context injected
    'rag_injection_fraction': 0.30,

    # Held-out eval set size
    'eval_holdout_size': 200,

    # Override save dirs to Drive for persistence across sessions
    # 'train_save_dir': '/content/drive/MyDrive/emotwen/data/sft_train',
    # 'val_save_dir':   '/content/drive/MyDrive/emotwen/data/sft_val',
    # 'eval_save_dir':  '/content/drive/MyDrive/emotwen/data/eval_200',
}

In [ ]:
# ── Run pipeline ──────────────────────────────────────────────────────────────
from src.data_prep import run

results = run(CONFIG)
print('\n── Results ──────────────────────────────────────────────')
for k, v in results.items():
    if not isinstance(v, dict):
        print(f'  {k}: {v}')
print('\nSources breakdown:')
for k, v in results['sources_breakdown'].items():
    print(f'  {k}: {v}')

In [ ]:
# ── Inspect sample conversations ──────────────────────────────────────────────
from datasets import load_from_disk
import os

train_ds = load_from_disk(os.path.join(REPO_DIR, 'data', 'sft_train'))

print(f'Train set size: {len(train_ds)}')
print('\n── Sample conversations ─────────────────────────────────')
for i in [0, 100, 500]:
    ex = train_ds[i]
    print(f'\n[Example {i}] source={ex["source"]}')
    for msg in ex['messages']:
        role = msg['role'].upper()
        content = msg['content'][:200].replace('\n', ' ')
        print(f'  {role}: {content}')
    print()